# Server metrics views

This notebook mirrors `examples/03_server_metrics.py` and demonstrates how to simulate
server metrics with trend, seasonality, and anomalies while exposing multiple unit views.

Requirements:
- `torch` and `toyts`
- `matplotlib` for plotting (skip the plotting cell if you do not have it installed)


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from toyts.core.pipeline import SynthPipeline
from toyts.processes.trend_season import TrendSeasonAnomalyProcess
from toyts.views.noise import NormalizeView
from toyts.views.units import (
    ClippingView,
    UnitsAbsoluteView,
    UnitsPercentOfCapacityView,
)


## Device and reproducibility


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rng = torch.Generator(device=device).manual_seed(42)


## Define the latent process


In [ ]:
process = TrendSeasonAnomalyProcess(
    seq_len=720,
    components=3,
    regime_classes=["steady", "ramping", "spiky"],
    anomaly_classes=["none", "drop", "spike"],
    slope_max=0.5,
    season_amp=15.0,
    spiky_boost=2.5,
    season_freq_min=1.0,
    season_freq_max=3.0,
    anomaly_scale=40.0,
)


## Define observed views

Each view converts the same latent process into a different unit representation.


In [ ]:
views = {
    "absolute": UnitsAbsoluteView(),
    "percent_of_capacity": UnitsPercentOfCapacityView(
        capacity_min=80.0, capacity_max=120.0
    ),
    "cpu_percent": torch.nn.Sequential(
        UnitsPercentOfCapacityView(capacity_min=90.0, capacity_max=110.0),
        ClippingView(min_value=0.0, max_value=100.0),
    ),
    "normalized": torch.nn.Sequential(
        UnitsPercentOfCapacityView(capacity_min=90.0, capacity_max=110.0),
        NormalizeView(),
    ),
}


## Run the pipeline and inspect outputs


In [ ]:
pipeline = SynthPipeline(process=process, views=views)
pipeline.to(device)

batch = pipeline(batch_size=4, device=device, rng=rng)

print("Generated batch keys:", batch.keys())
for name, obs in batch.items():
    signal = obs.x  # [B, C, L]
    regime_labels = obs.y["regime"]  # [B]
    anomaly_labels = obs.y["anomaly_type"]  # [B]

    print(f"--- View: {name} ---")
    print("  Signal shape:", signal.shape)
    print("  Regime labels:", regime_labels)
    print("  Anomaly labels:", anomaly_labels)
    print(f"  Signal range: [{signal.min():.2f}, {signal.max():.2f}]")

capacity_meta = batch["percent_of_capacity"].view_meta("UnitsPercentOfCapacityView")
print("\nView metadata:")
print("  percent_of_capacity capacity:", tuple(capacity_meta["capacity"].shape))


## Plot the views

This plots one sample across all unit views and saves the figure under `examples/figures`.


In [ ]:
output_dir = Path("figures")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "03_server_metrics.png"

fig, axes = plt.subplots(len(views), 1, figsize=(12, 10), sharex=True, sharey=False)
for axis, (name, obs) in zip(axes, batch.items()):
    signal = obs.x[0, 0, :].detach().cpu()  # [L]
    axis.plot(signal.numpy())
    axis.set_title(f"View: '{name}'")
    axis.grid(True)
plt.tight_layout()
fig.savefig(output_path, dpi=150, bbox_inches="tight")
plt.show()

print("saved figure", output_path)
